# Room Reconstruction — independent VGGT reconstruction

In VS Code, open this notebook and select **Select Kernel → Colab → New Colab Server**, choosing a GPU server.
Put `roomdelta3d_colab_submission.zip` in **My Drive / Room Reconstruction** before running the first cell.
The first cell mounts Drive, copies the ZIP to local runtime storage, and verifies its contents.
It contains only the reconstruction code/configuration and the 32 Before + 30 After selected PNGs/model inputs.
No videos, Gemini credentials, VLM prompts, evaluation labels or model checkpoints are included.

Run cells in order. Pilot outputs are separate from full predictions. All coordinates remain relative, and successful inference does not establish real-scene accuracy.
The setup uses an isolated Python environment with pinned PyTorch/torchvision and a content-verified VGGT checkpoint.


In [ ]:
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive
from IPython.display import display, Image as DisplayImage

EXPECTED_PROJECT_FILES = {'reconstruction/alignment.py': '364d33e8ea72475cbfbbd1a16f9ca9886ba4854e305b2ab4c67803541759591f', 'reconstruction/archive.py': 'f7f77fd6257ea1e6450dd5222f637b489858da62d94c92a873a1aaeee30a2553', 'reconstruction/change_detection.py': 'b5240391a7720a4ebc98e498f3f3ffce6442175e04962ef64af3755c583d10fa', 'reconstruction/change_report.py': 'b40b0bbbcc616707316223e30d0579a8955eff6c10c0d8cec70754a41600e20c', 'reconstruction/change_review.py': '5ed78e25deb9e9e8152836476f13704af094366fe21539338437e24f68ffa9c2', 'reconstruction/change_stage.py': '313347fa4d8ee1bb27a5befd087736d2da316f385378074fffc4b36988ac34f4', 'reconstruction/comparison.py': '9084680beacb66de1ee20dfdc117b58793b53555a3339c64b82446a5e49cddc1', 'reconstruction/comparison_viewer.py': 'cb08c7dcf35cc903fe4c7e78fd31ccb19cf1ee0de7c7952b94003b1d9109577c', 'reconstruction/core.py': '575188a0196794e021fe8c5631b29b7fe6ef578eed02f794cc913feb06c4dc9c', 'reconstruction/inference.py': 'a764dc3d60fd6e3d328cabdb7d48200a817d70fd9ec7ca1f41227ccc6f679463', 'reconstruction/inspection_viewer.py': '2108277171a29a1fe1e956e84396dd697c89f8c6af171bf511b131a5cd54889d', 'reconstruction/object_archive.py': '0d18d0cf788704715ea785d825a7e0e3740fdb99f667677616f38a25d91ae124', 'reconstruction/object_association.py': '63defe8e89e20fb72c5cc566b3a46a0ad1f59e9b4435acc851aad161830d6395', 'reconstruction/object_contracts.py': 'dc55af1a0620a3395dbcdb7182527643f11cd22b49be7c027809f81bbb90845c', 'reconstruction/object_geometry.py': 'a366e3d9399d6cb481e7e88338054a8af942d152a17eb52d29c966a3887b1091', 'reconstruction/object_inspection.py': '4d9fbd89f6be7ed03704f5d3f38bb48bc8fac259ed53487c4f83fa677812f4d4', 'reconstruction/object_models.py': '5c983c113e8f5058ae43836eeba0eeb6e6b3ac9db515b776e80b42b5b27cfed7', 'reconstruction/object_pipeline.py': '4b15865dbed79c0b65f03a9590341b1eeae51e584bf05569fb4ae55eaef68299', 'reconstruction/refined_inventory.py': '97fe79f2d49b412758abdee2c042ce108dac163ec51595ad9a9e52069cb2ff59', 'reconstruction/refinement.py': 'f5bf4e39f25a30b2f01384c3662c37ba786e0fd2178809bd9b7ada59c742c59e', 'reconstruction/refinement_evidence.py': 'ae5bf7cd818b38242b17f15aa1eaca03f362992d6fbed517588db98c44b897c9', 'reconstruction/refinement_gemini.py': '1605d86da2a092c46b36cab937069ab66a88bea3234075f86b17c3f6ffd4e5fc', 'reconstruction/region_reference.py': 'edff0269ee47ab0690a6d23fcb0144f44377e3e31d5c142a5ab5897c115bc0b4', 'reconstruction/report_evidence.py': '54d7da9de056976ed255f9f01ffb34c7ef400c8588ea02aa7df9186c22414435', 'reconstruction/report_render.py': 'e0381a13d27e0fae4a9e1442309ca4b01c8e11ad578313291f7e81e5c62b864e', 'reconstruction/review_analysis.py': 'b632c913f5dfc2b9e0670e1f5f7d6d612b1367da2ae4b0a1e28d20b7b3a73d5b', 'reconstruction/review_config.py': '565da942a109caf28eff94762d2e9615254431040f92c5ce1b9b18e49647b97d', 'reconstruction/review_evidence.py': 'be5b12c6e0e91fb2e72073916f19c7f633f99d965bfc02349adda11ef737c1b4', 'reconstruction/static_alignment.py': '3b03ef4e7def75c659112ff1f43081800e4b7508434193a04617d4a1255dc1c4', 'reconstruction/sync.py': 'b258e0a595352dff0b4ab263857784dad3f81e29d692ca50478b53e9fc830e4d', 'reconstruction/sync_media.py': '8569dbee197a21f953ddcf3dd04179b1e37d2589f73f464253b7295df95cdf72', 'reconstruction/sync_viewer.py': 'f879240d0de65b61f9acd2582fd15ed411fdde0d9fe40e3ff54c9dda1265644e', 'reconstruction/viewer.py': '8298d87a39fca5514428d1627be897970b04178982eacd2ab61ae0bb1b5d96ac', 'reconstruction/__init__.py': 'b5b5494be0edc90e8697bd20327f81e3fb15139c75eb994b359f45a880a122b2', 'scripts/reconstruct.py': '2e1acba6d83c3fb8279f6b0d6193fb9cbc1a8d90b4f3ee45a5c5294e08f7826b', 'scripts/setup_colab.py': 'efb1b39b26ec1c9b22cb38ce12b963e2cb5c95b514d250ae2f919a0b606103fc', 'scripts/colab_run.py': '81b41e38a23b5ec80057350f450f2aefffc0c4e384db2dbd139a0e1bc3fda7fc', 'configs/reconstruction.json': '0baeedb6141d3d27918aae43b4b6733ddc134de9f961f5efbf71693fb826d8c1', 'configs/models.lock.json': '830ec392f44e7fd8a50fda04720a8e034550df093c184bb0ba5d9edb52745585', 'requirements-reconstruction.txt': 'c0c5c566772177b1238d7a00b6d65a7c850b72ff053c75cfc701d464a1f65f1a', 'reference/vggt_load_fn.py': 'd57b7a16c2e04333a2e68d365aa21f1dd61759e984f744cbcbe6b66d787f37ef'}

def verify_active_project():
    changed = []
    for relative, expected in EXPECTED_PROJECT_FILES.items():
        path = PROJECT / relative
        if not path.is_file():
            changed.append(relative)
            continue
        with path.open('rb') as handle:
            if hashlib.file_digest(handle, 'sha256').hexdigest() != expected:
                changed.append(relative)
    if changed:
        raise RuntimeError('Notebook/bundle code mismatch: ' + ', '.join(changed) +
                           '. Upload the ZIP paired with this notebook and extract it in a fresh runtime. '
                           'Read saved diagnostics before leaving the current runtime.')
    print('Active project matches this notebook:', len(EXPECTED_PROJECT_FILES), 'verified files')

# Edit this folder only if you put the ZIP elsewhere in My Drive.
DRIVE_FOLDER = Path('/content/drive/MyDrive/Room Reconstruction')
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')  # Complete the extension's Google authorization prompt.
drive_bundle = DRIVE_FOLDER / 'roomdelta3d_colab_submission.zip'
if not drive_bundle.is_file():
    raise FileNotFoundError(f'Put the prepared ZIP here, or edit DRIVE_FOLDER: {drive_bundle}')
BUNDLE_ROOT = Path('/content/roomdelta3d_bundle')
if BUNDLE_ROOT.exists():
    raise RuntimeError('Bundle directory already exists. Reuse the subsequent cells on this runtime, or start a fresh runtime.')
bundle_path = Path('/content/roomdelta3d_colab_ready.zip')
shutil.copyfile(drive_bundle, bundle_path)
BUNDLE_ROOT.mkdir()
with zipfile.ZipFile(bundle_path) as bundle:
    seen = set()
    for info in bundle.infolist():
        name = info.filename
        parts = PurePosixPath(name)
        if name in seen or parts.is_absolute() or '..' in parts.parts or '\\' in name or ':' in name:
            raise ValueError('Unsafe/duplicate bundle member')
        seen.add(name)
        if ((info.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('Symlink bundle member')
        if not info.is_dir():
            target = (BUNDLE_ROOT / name).resolve()
            if not target.is_relative_to(BUNDLE_ROOT):
                raise ValueError('Member escapes bundle root')
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(info) as source, target.open('wb') as destination:
                shutil.copyfileobj(source, destination)
receipt = json.loads((BUNDLE_ROOT / 'bundle_receipt.json').read_text())
actual = {p.relative_to(BUNDLE_ROOT).as_posix() for p in BUNDLE_ROOT.rglob('*') if p.is_file()}
assert actual == set(receipt['files']) | {'bundle_receipt.json'}
for name, expected in receipt['files'].items():
    with (BUNDLE_ROOT / name).open('rb') as handle:
        assert hashlib.file_digest(handle, 'sha256').hexdigest() == expected, name
PROJECT = BUNDLE_ROOT / 'project'
WORKSPACE = Path('/content/roomdelta3d_workspace')
RUN = WORKSPACE / 'reconstruction'
verify_active_project()
print('Verified Drive bundle:', drive_bundle, 'Files:', len(receipt['files']))


## Install pinned runtime and checkpoint
The checkpoint download is approximately 5 GB. This cell records package versions and verifies the checkpoint SHA-256.
The system Python/driver/GPU assignment is recorded rather than assumed reproducible by Colab.


In [ ]:
import os
from collections import deque

def run_logged(command, label):
    # Pipe child output explicitly so VS Code receives it through notebook stdout.
    log_directory = WORKSPACE / 'logs'
    log_directory.mkdir(parents=True, exist_ok=True)
    log_path = log_directory / f'{label}.log'
    tail = deque(maxlen=60)
    print(f'{label}: starting; log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED='1')) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{label} failed (exit {returncode}). Log: {log_path}\nLast output:\n' + ''.join(tail))
    print(f'{label}: completed', flush=True)

verify_active_project()
run_logged([sys.executable, str(PROJECT / 'scripts/setup_colab.py'), '--workspace', str(WORKSPACE)], 'setup')
PYTHON = str(WORKSPACE / 'venv/bin/python')
CLI = str(PROJECT / 'scripts/reconstruct.py')
if not RUN.exists():
    run_logged([PYTHON, CLI, 'import', '--archive', str(BUNDLE_ROOT / 'prepared_run.zip'), '--output', str(RUN)], 'import')
print('Prepared run:', RUN)


## Measured GPU pilots
Run 4, 8, then 16 representative frames per capture in fresh, separate workers.
Each worker checks the saved model tensors against the exact upstream loader before inference.
Attention is evaluated in bounded query slices while retaining all frame keys/values. Runtime records include this policy and memory measurements.
Inspect failures/logs instead of silently reducing frame count or resolution. A passed pilot does not guarantee the full run will fit.
If a pilot has already failed in this runtime, run the next diagnostic cell to inspect its saved worker exceptions without rerunning inference.


In [ ]:
# Read-only diagnostics: safe to run after a failure, including with an older bundle.
from collections import deque

def show_pilot_diagnostics():
    def read_saved_json(path):
        try:
            return json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError) as error:
            print(f'Cannot read {path}: {error}')
            return {}

    report_path = RUN / 'pilot_report.json'
    report = read_saved_json(report_path) if report_path.is_file() else {}
    print('Saved pilot report:', report.get('status', 'not available'), report_path)
    pilots = report.get('pilots', [])
    if not pilots:
        # Workers may have failed before the parent wrote its report.
        pilots = [{'requested_frames': int(path.name[1:]), 'captures': {}}
                  for path in sorted((WORKSPACE / 'pilots').glob('p[0-9][0-9]'))]
    for pilot in pilots:
        count = pilot['requested_frames']
        target = WORKSPACE / 'pilots' / f'p{count:02d}'
        for capture in ('before', 'after'):
            measurement = pilot.get('captures', {}).get(capture, {})
            env_path = target / capture / 'inference_environment.json'
            env = measurement.get('environment') or (read_saved_json(env_path) if env_path.is_file() else {})
            status = measurement.get('status', env.get('status', 'unknown'))
            def gib(key):
                value = env.get(key)
                return round(value / 1024**3, 3) if value is not None else 'unavailable'
            print(count, capture, status, 'allocated GiB:', gib('cuda_peak_allocated_bytes'),
                  'reserved GiB:', gib('cuda_peak_reserved_bytes'), 'inference seconds:', env.get('inference_sec'))
            print('Stage:', env.get('stage', 'unknown'), 'attention:', env.get('attention', 'not recorded'))
            if status == 'completed':
                continue
            print('Worker error:', env.get('error') or measurement.get('worker_error')
                  or measurement.get('error') or 'No structured worker error; inspect log below.')
            log_path = target / 'logs' / f'{capture}.log'
            print('Worker log:', log_path)
            try:
                with log_path.open(encoding='utf-8', errors='replace') as handle:
                    tail = ''.join(deque(handle, maxlen=60))
            except OSError as error:
                tail = measurement.get('worker_log_tail') or f'Worker log unavailable: {error}'
            print(tail or '(empty worker log)')

show_pilot_diagnostics()


In [ ]:
verify_active_project()
try:
    run_logged([PYTHON, str(PROJECT / 'scripts/colab_run.py'), '--run', str(RUN), '--workspace', str(WORKSPACE), '--phase', 'pilot'], 'pilot')
finally:
    show_pilot_diagnostics()
for name in ('before', 'after'):
    panels = sorted((WORKSPACE / 'pilots/p04' / name / 'previews').glob('*_panel.png'))
    if panels:
        print(name, 'actual 4-frame pilot prediction; independent pilot coordinate/scale frame')
        display(DisplayImage(filename=str(panels[0]), width=1100))
    else:
        print(name, '4-frame pilot preview unavailable; inspect worker diagnostics above.')


## Full independent predictions
Process all 32 Before and 30 After selected frames separately. Save depth/confidence/masks, cameras, colored PLY/numerical clouds, all point observations, and quality reports.
`--resume` reuses only hash-compatible, structurally verified captures. Never use pilot geometry as full-run geometry.


In [ ]:
verify_active_project()
run_logged([PYTHON, str(PROJECT / 'scripts/colab_run.py'), '--run', str(RUN), '--workspace', str(WORKSPACE), '--phase', 'full'], 'full')
run_logged([PYTHON, CLI, 'verify', '--run', str(RUN)], 'verify')
for name in ('before', 'after'):
    report = json.loads((RUN / name / 'quality_report.json').read_text())
    print(name, 'points:', report['cloud_points'], 'internal consistency:', report['internal_consistency'],
          'real-scene accuracy:', report['real_scene_accuracy'])
    panels = sorted((RUN / name / 'previews').glob('*_panel.png'))
    display(DisplayImage(filename=str(panels[0]), width=1100))


## Save portable results to Drive
The ZIP contains source/model images, depth arrays, camera records, cloud/provenance and reports; it excludes the checkpoint and environment installation.
This cell copies the completed archive directly beside the input ZIP in Drive and verifies its SHA-256 after copying.
Download that ZIP from Drive, import it locally using `scripts/reconstruct.py import`, then run `verify`, `preview`, or `view`.
Drive persistence occurs after this cell finishes; interrupted inference stays in the current runtime until explicitly saved.


In [ ]:
RESULTS = WORKSPACE / 'roomdelta3d_reconstruction.zip'
if RESULTS.exists():
    raise RuntimeError('Results archive already exists; choose a new RESULTS filename to preserve it.')
run_logged([PYTHON, CLI, 'pack', '--run', str(RUN), '--archive', str(RESULTS)], 'pack')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive is no longer mounted; remount it before copying the local results archive.')
saved_directory = DRIVE_FOLDER
saved_archive = saved_directory / RESULTS.name
staging_archive = saved_archive.with_suffix('.zip.partial')
receipt_path = saved_archive.with_suffix('.receipt.json')
if any(path.exists() for path in (saved_archive, staging_archive, receipt_path)):
    raise RuntimeError(f'Drive result or receipt already exists; choose a new RESULTS filename to preserve it: {saved_archive}')
shutil.copyfile(RESULTS, staging_archive)
with RESULTS.open('rb') as handle:
    expected = hashlib.file_digest(handle, 'sha256').hexdigest()
with staging_archive.open('rb') as handle:
    actual = hashlib.file_digest(handle, 'sha256').hexdigest()
if actual != expected:
    raise RuntimeError('Drive copy failed SHA-256 verification; local results remain available.')
staging_archive.rename(saved_archive)
save_receipt = {'archive': str(saved_archive), 'sha256': actual, 'bytes': saved_archive.stat().st_size}
receipt_path.write_text(json.dumps(save_receipt, indent=2) + '\n')
print('Saved verified results to Drive:', saved_archive)
print('SHA-256:', actual)
